# Medication Sales Across Pharmacies – Data Cleaning

This dataset sourced from a simulated POS (Point of Sale) database, contains detailed information on customers, products, and sales transactions across multiple pharmacy locations.

**Importación de librerías**  

Se importan las librerías necesarias para análisis de datos y estadística:  
- `pandas` para manipulación de datos en DataFrames.  

In [1]:
import pandas as pd

**Carga del dataset de pharmacies**

Load the pharmacy sales CSV file into a DataFrame, using error 
handling to manage cases where the file may not exist or cannot be found.

In [2]:
try:
  pharmacy = pd.read_csv("pharmacies_sales.csv")
  pharmacy_copy = pharmacy.copy()
  print("File was loaded with success and save as dataframe!")
  print(f"Count of rows and columns: {pharmacy_copy.shape}")
except FileNotFoundError:
  print("Error: pharmacies_sales.csv was not found.")

File was loaded with success and save as dataframe!
Count of rows and columns: (1000, 17)


**Información del DataFrame `pharmacy_copy`**

Display the structure of the DataFrame, including column names, data types, and non-null counts.

In [3]:
print("Information about the dataframe: ")
pharmacy_copy.info()

Information about the dataframe: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            1000 non-null   object 
 1   Year            1000 non-null   int64  
 2   Month_Name      1000 non-null   object 
 3   Month           1000 non-null   int64  
 4   Day             1000 non-null   int64  
 5   Season          1000 non-null   object 
 6   Location_Store  1000 non-null   object 
 7   Gender          1000 non-null   object 
 8   Age             1000 non-null   int64  
 9   Type_Plan       1000 non-null   object 
 10  City_Client     1000 non-null   object 
 11  Name_Product    1000 non-null   object 
 12  Category        1000 non-null   object 
 13  Covered_Plan    1000 non-null   object 
 14  Unit_Price      1000 non-null   float64
 15  Quantity        1000 non-null   int64  
 16  Total_Sale      995 non-null    float64
dtype

**Eliminación de duplicados `df_pharmacy`**

Remove any duplicated rows from the dataset to ensure data integrity.

In [4]:
# Remove duplicates
df_pharmacy = pharmacy_copy.drop_duplicates().copy()

# Check for duplicates
duplicates = pharmacy_copy[pharmacy_copy.duplicated()]

if duplicates.empty:
    print("No repeated data detected.")
else:
    print("Repeated data found:", duplicates)

No repeated data detected.


**Transform your data**

Data transformation is the process of converting data from one format while 
preserving the cleaning and integrity of the dataset. This process 
makes the data more organized, consistent and useful for analysis.

In [5]:
def fix_season(df, month, season):
    """ Correct the Season column for a specific month. """
    idx = df.query(f"Month == {month} and Season != '{season}'").index
    df.loc[idx, 'Season'] = season
    return df

def capitalize_lowercase_season(df):
    """ Capitalize the first letter of Season only if it is all lowercase. """
    lowercase_fix = df['Season'].str.islower()
    capitalized = df.loc[lowercase_fix, 'Season'].str.capitalize()
    df.loc[lowercase_fix, 'Season'] = capitalized
    return df

def fix_location_store(df):
    """ Standardizes the Location Store column in title case. """
    lowercase_fix = df['Location_Store'] != df['Location_Store'].str.title()
    title_word = df.loc[lowercase_fix, 'Location_Store'].str.title()
    df.loc[lowercase_fix, 'Location_Store'] = title_word
    return df

def fix_gender(df):
    """ Standardizes the Gender by converting all lowercase entries to uppercase. """
    lowercase_fix = df['Gender'].str.islower()
    uppercase_word = df.loc[lowercase_fix, 'Gender'].str.upper()
    df.loc[lowercase_fix, 'Gender'] = uppercase_word
    return df

def fix_city_client(df):
    """ Standardizes the City Client to title case. """
    lowercase_fix = df['City_Client'] != df['City_Client'].str.title()
    title_word = df.loc[lowercase_fix, 'City_Client'].str.title()
    df.loc[lowercase_fix, 'City_Client'] = title_word
    return df

def fix_product_units(df, pattern):
    """Cleans the Name Product by appling a regex replacement. """
    df['Name_Product'] = df['Name_Product'].str.replace(pattern, r'\1 ', regex=True)
    return df

def update_category(df, product_name, new_category):
    """ Updates the Category for a specific product. """
    df.loc[df['Name_Product'] == product_name, 'Category'] = new_category
    return df

def update_total_sale(df):
    """ Recalculates and updates the Total Sale. """
    df['Total_Sale'] = df['Total_Sale'].fillna(0)
    df['Total_Sale'] = (df['Quantity'] * df['Unit_Price']).round(2)
    return df

**Data Cleaning for Pharmacy Sales Dataset**

Steps performed:
1. Fix incorrect or inconsistent season values.
2. Standardize season names to proper capitalization.
3. Correct inconsistent store location names.
4. Format gender labels to uppercase.
5. Standardize customer city names to title case.
6. Clean product unit formatting using regex.
7. Reassign product categories where misclassified.
8. Recalculate total sale values based on quantity and price.
9. Save the cleaned dataset to a new CSV file.

In [6]:
try:
    df_pharmacy = fix_season(df_pharmacy, 6, 'Summer')
    df_pharmacy = fix_season(df_pharmacy, 9, 'Fall')
    df_pharmacy = fix_season(df_pharmacy, 12, 'Winter')

    df_pharmacy = capitalize_lowercase_season(df_pharmacy)

    df_pharmacy = fix_location_store(df_pharmacy)

    df_pharmacy = fix_gender(df_pharmacy)

    df_pharmacy = fix_city_client(df_pharmacy)

    df_pharmacy = fix_product_units(df_pharmacy, r'(\d+)(?=mg\b|ml\b)')

    df_pharmacy = update_category(df_pharmacy, 'Protector Solar', 'Cuidado personal')
    df_pharmacy = update_category(df_pharmacy, 'Venda Elastica', 'Primeros auxilios')
    df_pharmacy = update_category(df_pharmacy, 'Curitas Adhesivas', 'Primeros auxilios')
    df_pharmacy = update_category(df_pharmacy, 'Cepillos de dientes', 'Cuidado personal')
    df_pharmacy = update_category(df_pharmacy, 'Gel Antibacterial', 'Cuidado personal')
    df_pharmacy = update_category(df_pharmacy, 'Condroitina y Glucosamina', 'Suplementos')
    df_pharmacy = update_category(df_pharmacy, 'Panales para adultos', 'Cuidado personal')

    df_pharmacy = update_category(df_pharmacy, 'Jarabe para la tos', 'Antigripales')
    df_pharmacy = update_category(df_pharmacy, 'Crema para picazon', 'Cuidado personal')
    df_pharmacy = update_category(df_pharmacy, 'Ibuprofeno 200 mg','Analgesico')

    df_pharmacy = update_total_sale(df_pharmacy)

except Exception as e:
    print(f"Error: {e}")

finally:
    df_pharmacy.to_csv('cleaned_pharmacies_sales.csv', index=False)
    print("The pharmacy sales data has been cleaned and saved to the CSV file.")

The pharmacy sales data has been cleaned and saved to the CSV file.
